#Predicting Survival From Titanic Crush

In [21]:
import pandas as pd

In [22]:
df= pd.read_csv("/content/titanic.csv")

In [23]:
df.head()

,PassengerId,Name,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,1,"Braund, Mr. Owen Harris",3,male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,"Heikkinen, Miss. Laina",3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,female,35.0,1,0,113803,53.1000,C123,S,1
4,5,"Allen, Mr. William Henry",3,male,35.0,0,0,373450,8.0500,NaN,S,0


EDA Analysis and Data Cleaning

In [24]:
#Drop irrelevant columns

df.drop(['PassengerId','Name','Parch','Ticket','Cabin','Embarked'], axis='columns', inplace=True)


In [25]:
df.head()

,Pclass,Sex,Age,SibSp,Fare,Survived
0,3,male,22.0,1,7.2500,0
1,1,female,38.0,1,71.2833,1
2,3,female,26.0,0,7.9250,1
3,1,female,35.0,1,53.1000,1
4,3,male,35.0,0,8.0500,0


In [26]:
#separate target variable
target = df.Survived
inputs= df.drop('Survived', axis='columns')


In [27]:
inputs.head()

,Pclass,Sex,Age,SibSp,Fare
0,3,male,22.0,1,7.2500
1,1,female,38.0,1,71.2833
2,3,female,26.0,0,7.9250
3,1,female,35.0,1,53.1000
4,3,male,35.0,0,8.0500


Encoding (One hot encoding)

In [28]:
dummies = pd.get_dummies(inputs.Sex)
dummies.head()

,female,male
0,False,True
1,True,False
2,True,False
3,True,False
4,False,True


In [29]:
# append dummies to input dataframe using the append method

inputs1= pd.concat([inputs,dummies], axis='columns')
inputs1.head(3)

,Pclass,Sex,Age,SibSp,Fare,female,male
0,3,male,22.0,1,7.2500,False,True
1,1,female,38.0,1,71.2833,True,False
2,3,female,26.0,0,7.9250,True,False


In [30]:
#Drop the sex column , not needed anymore
inputs1.drop('Sex', axis='columns', inplace=True)

Handle Missingness

In [31]:
inputs1.columns[inputs1.isna().any()]

Index(['Age'], dtype='object')

In [33]:
# Impute missing ages using mean
inputs1.Age = inputs1.Age.fillna(inputs1.Age.mean())
inputs1.head()

,Pclass,Age,SibSp,Fare,female,male
0,3,22.0,1,7.2500,False,True
1,1,38.0,1,71.2833,True,False
2,3,26.0,0,7.9250,True,False
3,1,35.0,1,53.1000,True,False
4,3,35.0,0,8.0500,False,True


Split data into train and split

In [36]:
from  sklearn.model_selection import train_test_split

In [37]:
X_train,X_test,y_train,y_test = train_test_split(inputs1,target,test_size=0.2)

In [38]:
len(X_train)

712

In [39]:
len(X_test)

179

Implement the Naive Bayes model

In [52]:
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()

In [53]:
model.fit(X_train,y_train)

GaussianNB()

In [54]:
model.score(X_test, y_test)

0.7821229050279329

Above score is not very high and can be improved through hyperparameter tuning

###Logistic Regression as a linear Benchmark

In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [56]:
log_reg = LogisticRegression(max_iter=300)

In [57]:
log_reg.fit(X_train,y_train)

LogisticRegression(max_iter=300)

In [58]:
y_log_pred = log_reg.predict(X_test)

In [59]:

print(f"Accuracy : {accuracy_score(y_test,y_log_pred)}")
print(f"Precision : {precision_score(y_test,y_log_pred)}")
print(f"Recall : {recall_score(y_test,y_log_pred)}")
print(f"f1 : {f1_score(y_test,y_log_pred)}")

Accuracy : 0.7597765363128491
Precision : 0.7413793103448276
Recall : 0.6056338028169014
f1 : 0.6666666666666666


Logistic Regression achieved an accuracy of 75.9%, with moderate precision (74.1%) but relatively low recall (60.6%). The reduced recall indicates that the model fails to identify a substantial portion of survivals, making it less suitable as a standalone classifier. This behavior is consistent with the partial class overlap observed in the PCA visualization and the limitations of linear decision boundaries.

### Random Forest

In [62]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [67]:
param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth' : [10,20],
    'criterion' : ['gini', 'entropy']
}

In [68]:
#instantiate our classifier
model = RandomForestClassifier(random_state=42)

In [81]:
grid_search = GridSearchCV(estimator=model,param_grid=param_grid,scoring='accuracy',cv=5,verbose=2)

In [82]:
grid_search.fit(X_train,y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END .....criterion=gini, max_depth=10, n_estimators=100; total time=   0.2s
[CV] END .....criterion=gini, max_depth=10, n_estimators=100; total time=   0.2s
[CV] END .....criterion=gini, max_depth=10, n_estimators=100; total time=   0.2s
[CV] END .....criterion=gini, max_depth=10, n_estimators=100; total time=   0.2s
[CV] END .....criterion=gini, max_depth=10, n_estimators=100; total time=   0.2s
[CV] END .....criterion=gini, max_depth=10, n_estimators=150; total time=   0.5s
[CV] END .....criterion=gini, max_depth=10, n_estimators=150; total time=   0.5s
[CV] END .....criterion=gini, max_depth=10, n_estimators=150; total time=   0.5s
[CV] END .....criterion=gini, max_depth=10, n_estimators=150; total time=   0.5s
[CV] END .....criterion=gini, max_depth=10, n_estimators=150; total time=   0.5s
[CV] END .....criterion=gini, max_depth=10, n_estimators=200; total time=   0.5s
[CV] END .....criterion=gini, max_depth=10, n_es

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [10, 20],
                         'n_estimators': [100, 150, 200]},
             scoring='accuracy', verbose=2)

In [83]:
grid_search.best_params_

{'criterion': 'entropy', 'max_depth': 10, 'n_estimators': 150}

In [84]:
grid_search.best_score_

np.float64(0.8315177779966513)

In [73]:
best_rf_model = grid_search.best_estimator_

In [74]:
y_pred_rf = best_rf_model.predict(X_test)

In [75]:
print(f"Accuracy : {accuracy_score(y_test,y_pred_rf)}")
print(f"Precision : {precision_score(y_test,y_pred_rf)}")
print(f"Recall : {recall_score(y_test,y_pred_rf)}")
print(f"f1 : {f1_score(y_test,y_pred_rf)}")

Accuracy : 0.8435754189944135
Precision : 0.8909090909090909
Recall : 0.6901408450704225
f1 : 0.7777777777777778


The Random Forest model significantly improved performance compared to linear and probabilistic classifiers, achieving an accuracy of 84.4% and an F1-score of 77.8%. However, recall remained moderate due to the conservative nature of the classifier. Further improvements were achieved by optimizing hyperparameters for recall, adjusting the classification threshold, and increasing the number of trees, thereby reducing false negatives and improving clinical relevance.